# Using REVE to classify EEG

In this tutorial, we train a classification head on the REVE model to demonstrate how it can be used as a powerfull off-the-shelf feature extractor.

## Loading REVE

To load the REVE model, go to the [Hugging Face collection](https://huggingface.co/collections/brain-bzh/reve) and select your model size (e.g. base).

The model is gated so you need to accept the terms of the form on the website. In general, you will need to authenticate before using the model. You can do this in the CLI here :

In [1]:
# !hf auth login

In [2]:
from transformers import AutoModel
import torch

model = AutoModel.from_pretrained("brain-bzh/reve-base", trust_remote_code=True, torch_dtype="auto")
pos_bank = AutoModel.from_pretrained("brain-bzh/reve-positions", trust_remote_code=True, torch_dtype="auto")

`torch_dtype` is deprecated! Use `dtype` instead!


flash_attn not found, install it with `pip install flash_attn` if you want to use it


In [3]:
# Show all pre-registered positions
print(pos_bank.get_all_positions())


['A1', 'A2', 'C3', 'C4', 'CZ', 'F3', 'F4', 'F7', 'F8', 'FP1', 'FP2', 'FZ', 'O1', 'O2', 'P3', 'P4', 'PZ', 'T3', 'T4', 'T5', 'T6', 'OZ', 'Cz', 'Fpz', 'Fz', 'P7', 'P8', 'Pz', 'T7', 'T8', 'C1', 'C2', 'C5', 'C6', 'CP1', 'CP2', 'CP3', 'CP4', 'CPz', 'FC1', 'FC2', 'FC3', 'FC4', 'FCz', 'P1', 'P2', 'POz', 'AFz', 'P5', 'P6', 'PO3', 'PO4', 'AF3', 'AF4', 'AF7', 'AF8', 'CP5', 'CP6', 'F1', 'F2', 'F5', 'F6', 'FC5', 'FC6', 'FT7', 'FT8', 'Fp1', 'Fp2', 'Iz', 'Oz', 'P10', 'P9', 'PO7', 'PO8', 'TP7', 'TP8', 'F10', 'F9', 'FT10', 'FT9', 'FTT10h', 'FTT9h', 'PO10', 'PO9', 'TP10', 'TP9', 'TPP10h', 'TPP8h', 'TPP9h', 'TTP7h', 'CCP1h', 'CCP2h', 'CCP3h', 'CCP4h', 'CCP5h', 'CCP6h', 'CPP1h', 'CPP2h', 'CPP3h', 'CPP4h', 'CPP5h', 'CPP6h', 'FCC1h', 'FCC2h', 'FCC3h', 'FCC4h', 'FCC5h', 'FCC6h', 'FFC1h', 'FFC2h', 'FFC3h', 'FFC4h', 'FFC5h', 'FFC6h', 'FTT7h', 'FTT8h', 'PPO1h', 'PPO2h', 'TTP8h', 'T10', 'T9', 'AFF1h', 'AFF2h', 'AFF5h', 'AFF6h', 'AFp1', 'AFp2', 'POO1', 'POO2', 'PO5', 'PO6', 'AFF1', 'AFF2', 'AFp3h', 'AFp4h', 'FFT7

In [4]:
# Inspect the model
print(model)

Reve(
  (transformer): TransformerBackbone(
    (layers): ModuleList(
      (0-21): 22 x ModuleList(
        (0): Attention(
          (norm): RMSNorm()
          (to_qkv): Linear(in_features=512, out_features=1536, bias=False)
          (to_out): Linear(in_features=512, out_features=512, bias=False)
          (attend): ClassicalAttention()
        )
        (1): FeedForward(
          (net): Sequential(
            (0): RMSNorm()
            (1): Linear(in_features=512, out_features=2722, bias=False)
            (2): GEGLU()
            (3): Linear(in_features=1361, out_features=512, bias=False)
          )
        )
      )
    )
  )
  (to_patch_embedding): Sequential(
    (0): Linear(in_features=200, out_features=512, bias=True)
  )
  (fourier4d): FourierEmb4D()
  (mlp4d): Sequential(
    (0): Linear(in_features=4, out_features=512, bias=False)
    (1): GELU(approximate='none')
    (2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (ln): LayerNorm((512,), eps=1e-05, el

At this stage, the last layer of the model is `Identity()`, we will replace it with a classifier layer suited for our case.
The dataset has 22 channels, and samples of ~4s (1001 timepoints at 250 Hz). The model output will be of size `[B, 22, 5, D]`, with `B` the batch size and `D` the hidden dimension (512 for the base model). We thus set the final layer to be of size `22*5*D`.

In [5]:
dim = 22 * 5 * 512

model.final_layer = torch.nn.Sequential(
    torch.nn.Flatten(),
    torch.nn.RMSNorm(dim),
    torch.nn.Dropout(0.1),
    torch.nn.Linear(dim, 4),
)

NB: in practice, when using REVE on your tasks, you will only need to load it from Huggingface and modify the last layer.

## Training scripts

You can adjust some of the training parameters here.

In [6]:
# Training parameters
batch_size = 64
n_epochs = 20
lr = 1e-3
positions = pos_bank(["Fz", "FC3", "FC1", "FCz", "FC2", "FC4", "C5", "C3", "C1", "Cz", "C2", "C4", "C6", "CP3", "CP1", "CPz", "CP2", "CP4", "P1", "Pz", "P2", "POz"])

In [7]:
from transformers import set_seed

set_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

[BCI Competition IV-2a](https://www.bbci.de/competition/iv/) is a motor imagery EEG dataset with 22 channels, 4 classes (left hand, right hand, feet, tongue), and 9 subjects. The data is loaded using [MOABB](https://moabb.neurotechx.com/).

In [8]:
from functools import partial
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery
from scipy.signal import butter, lfilter
import numpy as np
from torch.utils.data import Dataset, random_split

def butter_bandpass(lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    return butter(order, [low, high], btype="band")

paradigm = MotorImagery(n_classes=4, resample=250, fmin=8, fmax=30)
bci_dataset = BNCI2014_001()
X, y, metadata = paradigm.get_data(dataset=bci_dataset)

b, a = butter_bandpass(8, 30, 250)
X = lfilter(b, a, X, axis=-1)

label_map = {"left_hand": 0, "right_hand": 1, "feet": 2, "tongue": 3}
y = np.array([label_map[label] for label in y])

class BCIDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return {"data": self.X[idx], "labels": self.y[idx]}

n = len(y)
n_train = int(0.7 * n)
n_val = int(0.15 * n)
n_test = n - n_train - n_val
full_dataset = BCIDataset(X, y)
train_ds, val_ds, test_ds = random_split(full_dataset, [n_train, n_val, n_test])

def collate(batch, positions):
    x_data = torch.stack([x["data"] for x in batch])
    y_label = torch.tensor([x["labels"] for x in batch])
    positions = positions.repeat(len(batch), 1, 1)
    return {"sample": x_data,"label": y_label.long(),"pos": positions}
collate_fn = partial(collate, positions=positions)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

Choosing from all possible events


### Training functions

For simplicity, we only implement a basic training loop that does not include:
- model souping
- LoRA wrappers
- Channel Mixup
- Position augmentation
- Two stage fine-tuning
- Stable AdamW optimizer

The results might thus differ slightly from the paper.

In [9]:
from tqdm.auto import tqdm
from sklearn.metrics import balanced_accuracy_score, cohen_kappa_score, f1_score, roc_auc_score, average_precision_score
from sklearn.preprocessing import label_binarize

from functools import partial

def train_one_epoch(model, optimizer, loader):
    model.train()
    pbar = tqdm(loader, desc="Training", total=len(loader))

    for batch_data in pbar:
        data, target, pos = (
            batch_data["sample"].to(device, non_blocking=True),
            batch_data["label"].to(device, non_blocking=True),
            batch_data["pos"].to(device, non_blocking=True),
        )
        optimizer.zero_grad()
        with torch.amp.autocast(dtype=torch.float16, device_type="cuda" if torch.cuda.is_available() else "cpu"):
            output = model(data, pos)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        pbar.set_postfix({"loss": loss.item()})


def eval_model(model, loader):
    model.eval()

    y_decisions = []
    y_targets = []
    y_probs = []
    score, count = 0, 0
    pbar = tqdm(loader, desc="Evaluating", total=len(loader))
    with torch.inference_mode():
        for batch_data in pbar:
            data, target, pos = (
                batch_data["sample"].to(device, non_blocking=True),
                batch_data["label"].to(device, non_blocking=True),
                batch_data["pos"].to(device, non_blocking=True),
            )
            with torch.amp.autocast(
                dtype=torch.float16, device_type="cuda" if torch.cuda.is_available() else "cpu"
            ):
                output = model(data, pos)

            decisions = torch.argmax(output, dim=1)
            score += (decisions == target).int().sum().item()
            count += target.shape[0]
            y_decisions.append(decisions)
            y_targets.append(target)
            y_probs.append(output)

    gt = torch.cat(y_targets).cpu().numpy()
    pr = torch.cat(y_decisions).cpu().numpy()
    pr_probs = torch.softmax(torch.cat(y_probs).float(), dim=1).cpu().numpy()
    acc = score / count
    balanced_acc = balanced_accuracy_score(gt, pr)
    cohen_kappa = cohen_kappa_score(gt, pr)
    f1 = f1_score(gt, pr, average="weighted")

    auroc = roc_auc_score(gt, pr_probs, multi_class='ovr')
    auc_pr = average_precision_score(label_binarize(gt, classes=[0, 1, 2, 3]), pr_probs, average='macro')

    return acc, balanced_acc, cohen_kappa, f1, auroc, auc_pr

### Train the model

We freeze the backbone and only fine-tune the classification head.

The best classifier is used on the test set at the end of the training.

In [10]:
optimizer = torch.optim.AdamW(model.final_layer.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)


criterion = torch.nn.CrossEntropyLoss()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

best_val_acc = 0
best_final_layer = None

for epoch in range(n_epochs):
    print(f"Epoch {epoch + 1}/{n_epochs}")
    train_one_epoch(model, optimizer, train_loader)
    _, b_acc, _, _, _, _ = eval_model(model, val_loader)
    if b_acc > best_val_acc:
        best_val_acc = b_acc
        best_final_layer = model.final_layer.state_dict()
    print(f"Validation balanced accuracy: {b_acc:.4f}, best: {best_val_acc:.4f}")
    scheduler.step(b_acc)


model.final_layer.load_state_dict(best_final_layer)
acc, balanced_acc, cohen_kappa, f1, auroc, auc_pr = eval_model(model, test_loader)

# Results
print("acc", acc)
print("balanced_acc", balanced_acc)
print("cohen_kappa", cohen_kappa)
print("f1", f1)
print("auroc", auroc)
print("auc_pr", auc_pr)


Epoch 1/20


Training:   0%|          | 0/57 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/13 [00:00<?, ?it/s]

Validation balanced accuracy: 0.3396, best: 0.3396
Epoch 2/20


Training:   0%|          | 0/57 [00:00<?, ?it/s]

...

Training:   0%|          | 0/57 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/13 [00:00<?, ?it/s]

Validation balanced accuracy: 0.4545, best: 0.4719


Evaluating:   0%|          | 0/13 [00:00<?, ?it/s]

acc 0.43774069319640563
balanced_acc 0.43767720140735045
cohen_kappa 0.24629722487911454
f1 0.43434347118301025
auroc 0.7034416675869923
auc_pr 0.461332768638365
